In [1]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import kagglehub


/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv


In [2]:
Tele=pd.read_csv('/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv')
df=pd.DataFrame(Tele)
print(df[:5])


   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [3]:
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
df = df.dropna()
print(df.shape)
print(df.dtypes)

(7032, 21)
customerID           object
gender               object
SeniorCitizen         int64
Partner              object
Dependents           object
tenure                int64
PhoneService         object
MultipleLines        object
InternetService      object
OnlineSecurity       object
OnlineBackup         object
DeviceProtection     object
TechSupport          object
StreamingTV          object
StreamingMovies      object
Contract             object
PaperlessBilling     object
PaymentMethod        object
MonthlyCharges      float64
TotalCharges        float64
Churn                object
dtype: object


In [4]:
df = pd.get_dummies(df, drop_first=True, columns=['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod','Churn'])
print(df.shape)
print(df.head)
X=df.drop(['customerID','Churn_Yes'],axis=1)

y=df['Churn_Yes']
print(X.shape)
print(y.shape)

(7032, 32)
<bound method NDFrame.head of       customerID  SeniorCitizen  tenure  MonthlyCharges  TotalCharges  \
0     7590-VHVEG              0       1           29.85         29.85   
1     5575-GNVDE              0      34           56.95       1889.50   
2     3668-QPYBK              0       2           53.85        108.15   
3     7795-CFOCW              0      45           42.30       1840.75   
4     9237-HQITU              0       2           70.70        151.65   
...          ...            ...     ...             ...           ...   
7038  6840-RESVB              0      24           84.80       1990.50   
7039  2234-XADUH              0      72          103.20       7362.90   
7040  4801-JZAZL              0      11           29.60        346.45   
7041  8361-LTMKD              1       4           74.40        306.60   
7042  3186-AJIEK              0      66          105.65       6844.50   

      gender_Male  Partner_Yes  Dependents_Yes  PhoneService_Yes  \
0           Fa

In [5]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score,confusion_matrix
scaler= StandardScaler()
X_scaled=scaler.fit_transform(X)
X_train,X_test,y_train,y_test=train_test_split(X_scaled,y,test_size=0.2,random_state=42)
model=LogisticRegression(max_iter=1000)
model.fit(X_train,y_train)
y_pred=model.predict(X_test)
accuracy=accuracy_score(y_test,y_pred)
print(accuracy)
con=confusion_matrix(y_test,y_pred)
print(con)

0.7874911158493249
[[915 118]
 [181 193]]


In [6]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
svcmodel=SVC(random_state=42)
svcmodel.fit(X_train,y_train)
y_svc_pred=svcmodel.predict(X_test)
print(accuracy_score(y_test,y_svc_pred))
rf_model=RandomForestClassifier(random_state=42)
rf_model.fit(X_train,y_train)
y_rf_pred=rf_model.predict(X_test)
print(accuracy_score(y_test,y_rf_pred))

0.7810945273631841
0.7846481876332623


In [7]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# 1. Define the Cross-Validation strategy
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 2. Create the Industry-Standard Pipeline
rf_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RandomForestClassifier(random_state=42))
])

# 3. Evaluate using ROC-AUC (The Pro Metric)
# Note: We pass the original X and y. The pipeline handles scaling internally!
scores = cross_val_score(rf_pipe, X, y, cv=cv, scoring='roc_auc')

print(f"Random Forest Pipeline ROC-AUC: {scores.mean():.3f} (+/- {scores.std():.3f})")

Random Forest Pipeline ROC-AUC: 0.822 (+/- 0.004)
